# Tabela 6 — Forma de pagamento dos exames preventivos das mulheres, por decis de renda domiciliar per capita (%)

PNS 2013 e 2019 — Mulheres com 25 anos ou mais

Este notebook reproduz o script fornecido para construir a Tabela 6 seguindo o padrão metodológico das Tabelas 2 e 5 do projeto.

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from IPython.display import display

# -------------------------------------------------
# Setup do ambiente
# -------------------------------------------------
sys.path.append(str(Path("..").resolve()))
from service.pns_service import get_dataframe

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

In [2]:
# -------------------------------------------------
# Carregamento dos dados (gold)
# -------------------------------------------------
variables = [
    "renda_domiciliar_pc",
    "preventivo_fez",
    "mamografia_fez",
    "preventivo_cobertura",
    "mamografia_cobertura",
]

sources = ["2013", "2019"]

df = get_dataframe(
    variables=variables,
    sources=sources,
    filters={
        "sexo": "2",
        "idade": {"operador": ">=", "valor": 25}
    }
)

In [3]:
# -------------------------------------------------
# Limpeza e padronização de tipos
# -------------------------------------------------
df["renda_domiciliar_pc"] = pd.to_numeric(
    df["renda_domiciliar_pc"], errors="coerce"
)

for col in ["preventivo_fez", "mamografia_fez"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .replace({"nan": np.nan})
        .astype(float)
    )

# Remover observações sem renda
df = df.dropna(subset=["renda_domiciliar_pc"])

In [4]:
# -------------------------------------------------
# Construção dos decis de renda (por ano)
# -------------------------------------------------
labels_decis = [
    "1º Decil de renda",
    "2º Decil de renda",
    "3º Decil de renda",
    "4º Decil de renda",
    "5º Decil de renda",
    "6º Decil de renda",
    "7º Decil de renda",
    "8º Decil de renda",
    "9º Decil de renda",
    "10º Decil de renda",
]

# --------------------------------------------------------------------
# 'origem' (ano da PNS) é incluída automaticamente pelo get_dataframe

# Decis calculados separadamente por ano (distribuição interna da 
# renda domiciliar per capita)
# --------------------------------------------------------------------
df["decil_renda"] = (
    df.groupby("origem")["renda_domiciliar_pc"]
    .transform(
        lambda x: pd.qcut(
            x,
            q=10,
            labels=labels_decis
        )
    )
)

In [5]:
# -------------------------------------------------
# Função auxiliar – distribuição percentual
# -------------------------------------------------
def tabela_cobertura(df, col):
    dist = df[col].value_counts(normalize=True) * 100
    dist = dist.round(2)
    dist = dist.reindex(list(dist.index) + ["Total"])
    dist.loc["Total"] = 100.0
    return dist

In [6]:
# -------------------------------------------------
# Construção da Tabela 6 (por ano)
# -------------------------------------------------
tabelas = {}

for ano in sorted(df["origem"].unique()):
    df_ano = df[df["origem"] == ano]

    tabela = pd.concat(
        {
            "Papanicolau": (
                df_ano[df_ano["preventivo_fez"] == 1]
                .groupby("decil_renda")
                .apply(lambda x: tabela_cobertura(x, "preventivo_cobertura"))
            ),
            "Mamografia": (
                df_ano[df_ano["mamografia_fez"] == 1]
                .groupby("decil_renda")
                .apply(lambda x: tabela_cobertura(x, "mamografia_cobertura"))
            ),
        },
        axis=1
    )

    tabelas[ano] = tabela

In [7]:
# -------------------------------------------------
# Exibição final (formato artigo)
# -------------------------------------------------
for ano, tabela in tabelas.items():
    print(
        f"Tabela 6 – Forma de pagamento dos exames preventivos das mulheres "
        f"por decis de renda domiciliar per capita (%), PNS {ano}"
    )

    display(tabela)

Tabela 6 – Forma de pagamento dos exames preventivos das mulheres por decis de renda domiciliar per capita (%), PNS 2013


Papanicolau  Mamografia
decil_renda                                      
1º Decil de renda  SUS          84.40       73.62
                   Pagou         9.90       11.55
                   Plano         5.71       14.83
                   Total       100.00      100.00
2º Decil de renda  SUS          82.88       76.29
                   Pagou        11.92       14.08
                   Plano         5.19        9.63
                   Total       100.00      100.00
3º Decil de renda  SUS          77.47       72.09
                   Pagou        12.85       12.29
                   Plano         9.68       15.61
                   Total       100.00      100.00
4º Decil de renda  SUS          73.54       70.23
                   Pagou        14.18       13.49
                   Plano        12.28       16.28
                   Total       100.00      100.00
5º Decil de renda  SUS          69.43       65.94
                   Plano        16.83       21.15
                   Pagou        13.74       12.91
                   Total       100.00      100.00
6º Decil de renda  SUS          65.05       62.82
                   Plano        20.63       26.14
                   Pagou        14.32       11.04
                   Total       100.00      100.00
7º Decil de renda  SUS          56.07       53.68
                   Plano        28.46       31.67
                   Pagou        15.47       14.65
                   Total       100.00      100.00
8º Decil de renda  SUS          46.54       44.09
                   Plano        39.22       43.50
                   Pagou        14.24       12.42
                   Total       100.00      100.00
9º Decil de renda  Plano        57.39       62.77
                   SUS          29.56       26.46
                   Pagou        13.05       10.77
                   Total       100.00      100.00
10º Decil de renda Plano        78.01       82.56
                   SUS          11.11        9.42
                   Pagou        10.87        8.02
                   Total       100.00      100.00

Tabela 6 – Forma de pagamento dos exames preventivos das mulheres por decis de renda domiciliar per capita (%), PNS 2019


Papanicolau  Mamografia
decil_renda                                      
1º Decil de renda  SUS          83.43       80.38
                   Pagou        16.57       19.62
                   Total       100.00      100.00
2º Decil de renda  SUS          80.30       78.76
                   Pagou        19.70       21.24
                   Total       100.00      100.00
3º Decil de renda  SUS          75.89       76.36
                   Pagou        24.11       23.64
                   Total       100.00      100.00
4º Decil de renda  SUS          74.76       73.41
                   Pagou        25.24       26.59
                   Total       100.00      100.00
5º Decil de renda  SUS          71.46       73.24
                   Pagou        28.54       26.76
                   Total       100.00      100.00
6º Decil de renda  SUS          73.74       73.13
                   Pagou        26.26       26.87
                   Total       100.00      100.00
7º Decil de renda  SUS          66.35       66.91
                   Pagou        33.65       33.09
                   Total       100.00      100.00
8º Decil de renda  SUS          58.06       59.04
                   Pagou        41.94       40.96
                   Total       100.00      100.00
9º Decil de renda  SUS          50.80       48.90
                   Pagou        49.20       51.10
                   Total       100.00      100.00
10º Decil de renda Pagou        76.11       76.87
                   SUS          23.89       23.13
                   Total       100.00      100.00

In [8]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd

def exportar_tabela6_revista_final(tabelas, filename="Tabela_06_Final_Revista.pdf"):
    plt.rcParams['font.family'] = 'serif'
    
    with PdfPages(filename) as pdf:
        for ano in sorted(tabelas.keys()):
            fig, ax = plt.subplots(figsize=(14, 10)) # Aumentamos a largura para caber as colunas
            ax.axis('off')

            # --- 1. TRANSFORMAÇÃO DOS DADOS ---
            # O dataframe original está 'empilhado'. Vamos desempilhar para ter as categorias nas colunas.
            df_orig = tabelas[ano]
            # Unstack move o índice de categorias (SUS, Plano, etc) para as colunas
            df_pivot = df_orig.unstack(level=1)
            
            # Reordenar colunas para seguir o padrão: SUS -> Plano -> Pagou -> Total
            # (Trata o caso de 2019 que não tem 'Plano')
            cols_ordem = ['SUS', 'Plano', 'Pagou', 'Total']
            existentes = [c for c in cols_ordem if c in df_pivot.columns.get_level_values(1)]
            df_final = df_pivot.reindex(columns=existentes, level=1).reset_index()

            # --- 2. TÍTULO ---
            titulo = f"Tabela 6 – Forma de pagamento dos exames preventivos das mulheres, por decis de renda\nfamiliar per capita (PNS {ano}, em %)"
            plt.text(0.5, 0.95, titulo, ha='center', va='top', fontsize=12, fontweight='bold', transform=ax.transAxes)

            # --- 3. PREPARAÇÃO DA TABELA ---
            data = df_final.values.tolist()
            # Criar o sub-cabeçalho (SUS, Plano, Pagou, Total...)
            sub_header = ["Decil"] + (existentes * 2) # *2 porque é para Papanicolaou e Mamografia
            table_data = [sub_header] + data

            num_cols = len(sub_header)
            the_table = ax.table(cellText=table_data, 
                                 loc='center', 
                                 cellLoc='center',
                                 edges='open',
                                 bbox=[0.05, 0.15, 0.9, 0.55]) # Ajuste de largura

            the_table.auto_set_font_size(False)
            the_table.set_fontsize(9)

            # --- 4. CABEÇALHOS AGRUPADOS (Papanicolaou e Mamografia) ---
            y_group = 0.76 
            y_line = 0.73
            meio = (num_cols - 1) // 2
            
            # Cálculo de centro para os grupos
            pos_pap = 0.05 + (0.9 * (1 + meio/2) / num_cols)
            pos_mam = 0.05 + (0.9 * (1 + meio + meio/2) / num_cols)

            plt.text(0.35, y_group, "Papanicolaou", ha='center', fontweight='bold', fontsize=11, transform=ax.transAxes)
            plt.text(0.72, y_group, "Mamografia", ha='center', fontweight='bold', fontsize=11, transform=ax.transAxes)

            # --- 5. LINHAS DE ESTILO (ABNT) ---
            # Linha superior e inferior (grossas)
            ax.plot([0.05, 0.95], [0.82, 0.82], color='black', lw=1.5, transform=ax.transAxes) 
            ax.plot([0.05, 0.95], [0.15, 0.15], color='black', lw=1.5, transform=ax.transAxes)
            
            # Linha abaixo do cabeçalho
            ax.plot([0.05, 0.95], [0.70, 0.70], color='black', lw=1, transform=ax.transAxes)

            # Linhas curtas sob os grupos
            ax.plot([0.18, 0.52], [y_line, y_line], color='black', lw=0.8, transform=ax.transAxes) # Sob Pap
            ax.plot([0.57, 0.92], [y_line, y_line], color='black', lw=0.8, transform=ax.transAxes) # Sob Mam

            # --- 6. AJUSTES DAS CÉLULAS ---
            for i in range(len(table_data)):
                # Decil à esquerda
                the_table[(i, 0)].set_text_props(ha='left')
                if i == 0:
                    for j in range(num_cols):
                        the_table[(i, j)].set_text_props(fontweight='bold')
                for j in range(num_cols):
                    the_table[(i, j)].set_height(0.06)

            # --- 7. NOTA E FONTE ---
            footer_y = 0.11
            if ano == "2019":
                nota = "Nota: A categoria 'Plano' não está disponível para 2019 devido a mudanças no questionário do IBGE."
                plt.text(0.05, footer_y, nota, ha='left', fontsize=8, style='italic', transform=ax.transAxes)
                footer_y -= 0.03
            
            plt.text(0.05, footer_y, f"Fonte: Elaboração própria a partir dos dados da PNS/{ano}, IBGE.", 
                     ha='left', fontsize=9, transform=ax.transAxes)

            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
            
    print(f"Sucesso! Tabela 6 salva em '{filename}' no padrão de revista.")

# Executar
exportar_tabela6_revista_final(tabelas)

Sucesso! Tabela 6 salva em 'Tabela_06_Final_Revista.pdf' no padrão de revista.
